# UniVTG Most Replayed 평가 v2 — 후보군·K 민감도 분석

이 노트북은 `result_top50`의 **Top-50 후보**를 이용해 기존 MR nDCG@4 평가를 유지하면서 실패 원인을 분해합니다.

핵심 실험:

1. 공식 Top-4 MR 평가
2. Candidate Recall@4/10/20/50
3. 후보군 크기 N=4/10/20/50 민감도
4. K=1/2/4/6 민감도
5. K별 Fixed vs Individual 대응 비교
6. Query index별 선택·MR 점수 진단

평가 정답은 YouTube Most Replayed Heatmap만 사용하며 Preview 영상은 필요하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 경로와 실험 설정

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/'
    '0_ASAC_11기_DL_3조'
)

UNIVTG_ROOT = PROJECT_ROOT / 'code' / 'UniVTG'
RESULT_ROOT = UNIVTG_ROOT / 'result_top50'
EVAL_ROOT = UNIVTG_ROOT / 'eval_result_main_mostreplayed_v2'

HEATMAP_ROOT = (
    PROJECT_ROOT
    / '유튜브데이터'
    / '메타데이터'
)

REQUIRE_COMPLETE = False
EXPECTED_MODES = ('fixed', 'individual')
PRIMARY_MODE = 'individual'

CANDIDATE_NS = (4, 10, 20, 50)
K_VALUES = (1, 2, 4, 6)
OFFICIAL_K = 4

MR_TIOU_THRESHOLD = 0.5
BOOTSTRAP_ITERATIONS = 5000
RANDOM_BASELINE_TRIALS = 200
RANDOM_SEED = 42

CATEGORY_SETTINGS = {
    'game': {
        'analysis_clip_sec': 2.0,
        'before_sec': 2.5,
        'after_sec': 2.5,
        'min_gap_sec': 2.0,
    },
    'movie': {
        'analysis_clip_sec': 2.0,
        'before_sec': 5.0,
        'after_sec': 10.0,
        'min_gap_sec': 5.0,
    },
    'podcast': {
        'analysis_clip_sec': 2.0,
        'before_sec': 5.0,
        'after_sec': 10.0,
        'min_gap_sec': 5.0,
    },
}

for label, path in {
    '추론 결과 폴더': RESULT_ROOT,
    'Heatmap 루트': HEATMAP_ROOT,
}.items():
    if not path.exists():
        raise FileNotFoundError(f'{label}가 없습니다: {path}')

EVAL_ROOT.mkdir(parents=True, exist_ok=True)

print('=' * 80)
print('UniVTG MR 평가 v2 설정 완료')
print('=' * 80)
print('추론 결과:', RESULT_ROOT)
print('Heatmap:', HEATMAP_ROOT)
print('평가 결과:', EVAL_ROOT)
print('Candidate N:', CANDIDATE_NS)
print('K 민감도:', K_VALUES)
print('부분 완료 결과 평가:', not REQUIRE_COMPLETE)

## 2. 공통 함수와 결과 파일 자동 로드

In [ ]:
import re
import unicodedata
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display


def normalize_text(value) -> str:
    if pd.isna(value):
        return ''
    return unicodedata.normalize('NFC', str(value).strip())


def normalize_column_name(value) -> str:
    return normalize_text(value).lower().replace(' ', '_')


def canonical_category(value) -> str:
    key = (
        normalize_text(value).lower()
        .replace('-', '_').replace(' ', '_')
    )
    aliases = {
        'game': 'game',
        'gaming': 'game',
        '게임': 'game',
        'movie': 'movie',
        'movie_review': 'movie',
        'moviereview': 'movie',
        'drama': 'movie',
        '영화': 'movie',
        'podcast': 'podcast',
        '팟캐스트': 'podcast',
    }
    if key not in aliases:
        raise ValueError(f'지원하지 않는 category: {value!r}')
    return aliases[key]


def canonical_split(value) -> str:
    key = (
        normalize_text(value).lower()
        .replace('-', '_').replace(' ', '_')
    )
    aliases = {
        'training': 'train',
        'validation_a': 'val_a',
        'vala': 'val_a',
        'validation_b': 'val_b',
        'valb': 'val_b',
        'testa': 'test_a',
        'testb': 'test_b',
    }
    return aliases.get(key, key or 'unspecified')


def read_csv(path: Path, *, dtype=str) -> pd.DataFrame:
    frame = pd.read_csv(path, encoding='utf-8-sig', dtype=dtype)
    frame.columns = [
        normalize_column_name(column)
        for column in frame.columns
    ]
    return frame


def require_columns(frame: pd.DataFrame, required, label: str) -> None:
    missing = [
        column for column in required
        if column not in frame.columns
    ]
    if missing:
        raise ValueError(
            f'{label} 누락 컬럼: {missing}\n'
            f'현재 컬럼: {frame.columns.tolist()}'
        )


def save_csv_safely(frame: pd.DataFrame, path: Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(
        temporary,
        index=False,
        encoding='utf-8-sig',
    )
    temporary.replace(path)
    print(f'{path.name}: {len(frame):,}행 저장')
    return path


def load_result_table(
    master_name: str,
    per_run_pattern: str,
    *,
    required=True,
):
    # 배치가 진행 중이어도 평가할 수 있도록
    # master CSV보다 영상·모드별 per-run CSV를 우선 결합합니다.
    per_run_files = sorted(
        RESULT_ROOT.rglob(per_run_pattern)
    )

    if per_run_files:
        frames = []
        failed_files = []

        for path in per_run_files:
            try:
                frame = read_csv(path, dtype=str)
                frame['_source_result_csv'] = str(path)
                frames.append(frame)
            except Exception as error:
                failed_files.append({
                    'path': str(path),
                    'error': repr(error),
                })

        if not frames:
            raise RuntimeError(
                f'{per_run_pattern} 파일을 읽지 못했습니다.'
            )

        if failed_files:
            print(
                f'주의: 읽기 실패한 per-run 파일 '
                f'{len(failed_files)}개'
            )
            display(pd.DataFrame(failed_files).head())

        combined = pd.concat(
            frames,
            ignore_index=True,
            sort=False,
        )
        print(
            f'{master_name}: per-run {len(frames):,}개 파일에서 '
            f'{len(combined):,}행 로드'
        )
        return combined

    master_path = RESULT_ROOT / master_name
    if master_path.exists():
        frame = read_csv(master_path, dtype=str)
        frame['_source_result_csv'] = str(master_path)
        print(
            f'{master_name}: master CSV '
            f'{len(frame):,}행 로드'
        )
        return frame

    if required:
        raise FileNotFoundError(
            f'{master_name}과 {per_run_pattern}을 '
            '찾지 못했습니다.'
        )

    return pd.DataFrame()


def bootstrap_mean_ci(
    values,
    iterations=BOOTSTRAP_ITERATIONS,
    seed=RANDOM_SEED,
):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    means = np.empty(int(iterations), dtype=float)

    for index in range(int(iterations)):
        sample = rng.choice(
            values,
            size=len(values),
            replace=True,
        )
        means[index] = sample.mean()

    return (
        float(values.mean()),
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975)),
    )


summary_raw = load_result_table(
    'run_summary.csv',
    '*_run_summary.csv',
)
candidates_raw = load_result_table(
    'candidate_segments_top50.csv',
    '*_candidate_segments_top50.csv',
)
selected_raw = load_result_table(
    'selected_segments.csv',
    '*_selected_segments.csv',
    required=False,
)

require_columns(
    summary_raw,
    [
        'video_id', 'mode', 'status', 'query_hash',
        'body_start_sec', 'body_end_sec', 'segment_sec',
        'category', 'split', 'creator_name', 'title',
        'updated_at_utc',
    ],
    'run_summary',
)
require_columns(
    candidates_raw,
    [
        'video_id', 'mode', 'query_hash',
        'candidate_rank', 'anchor_clip_index',
        'start_full_sec', 'end_full_sec',
        'duration_sec', 'combined_score',
    ],
    'candidate_segments_top50',
)

for frame in (summary_raw, candidates_raw, selected_raw):
    if frame.empty:
        continue
    frame['video_id'] = frame['video_id'].map(
        normalize_text
    )
    frame['mode'] = frame['mode'].map(
        lambda value: normalize_text(value).lower()
    )
    frame['query_hash'] = frame['query_hash'].map(
        normalize_text
    )

summary_raw['status'] = summary_raw['status'].map(
    lambda value: normalize_text(value).lower()
)
summary_raw['category'] = summary_raw['category'].map(
    canonical_category
)
summary_raw['split'] = summary_raw['split'].map(
    canonical_split
)
summary_raw['creator_name'] = (
    summary_raw['creator_name'].map(normalize_text)
)
summary_raw['title'] = (
    summary_raw['title'].map(normalize_text)
)
summary_raw['_updated_at'] = pd.to_datetime(
    summary_raw['updated_at_utc'],
    errors='coerce',
    utc=True,
)

summary_latest = (
    summary_raw
    .sort_values([
        '_updated_at',
        '_source_result_csv',
    ])
    .drop_duplicates(
        ['video_id', 'mode'],
        keep='last',
    )
    .reset_index(drop=True)
)

summary_latest = summary_latest[
    summary_latest['mode'].isin(EXPECTED_MODES)
].copy()

for column in [
    'body_start_sec',
    'body_end_sec',
    'segment_sec',
    'candidate_count',
    'final_top_k',
    'top_k',
]:
    if column in summary_latest.columns:
        summary_latest[column] = pd.to_numeric(
            summary_latest[column],
            errors='coerce',
        )

for column in [
    'candidate_rank',
    'anchor_clip_index',
    'start_full_sec',
    'end_full_sec',
    'duration_sec',
    'combined_score',
    'best_query_index',
]:
    if column in candidates_raw.columns:
        candidates_raw[column] = pd.to_numeric(
            candidates_raw[column],
            errors='coerce',
        )

if not selected_raw.empty:
    for column in [
        'score_rank',
        'final_rank',
        'anchor_clip_index',
        'start_full_sec',
        'end_full_sec',
        'duration_sec',
        'combined_score',
        'best_query_index',
    ]:
        if column in selected_raw.columns:
            selected_raw[column] = pd.to_numeric(
                selected_raw[column],
                errors='coerce',
            )

success_summary = summary_latest[
    summary_latest['status'].eq('success')
    & summary_latest['query_hash'].ne('')
    & summary_latest['body_start_sec'].notna()
    & summary_latest['body_end_sec'].notna()
    & (
        summary_latest['body_end_sec']
        > summary_latest['body_start_sec']
    )
].copy()

success_keys = success_summary[
    ['video_id', 'mode', 'query_hash']
].drop_duplicates()

candidates = candidates_raw.merge(
    success_keys,
    on=['video_id', 'mode', 'query_hash'],
    how='inner',
    validate='many_to_one',
)

candidates = (
    candidates
    .dropna(subset=[
        'candidate_rank',
        'anchor_clip_index',
        'start_full_sec',
        'end_full_sec',
        'duration_sec',
        'combined_score',
    ])
    .sort_values([
        'video_id',
        'mode',
        'query_hash',
        'candidate_rank',
        '_source_result_csv',
    ])
    .drop_duplicates(
        [
            'video_id',
            'mode',
            'query_hash',
            'candidate_rank',
        ],
        keep='last',
    )
    .reset_index(drop=True)
)

candidate_counts = (
    candidates
    .groupby([
        'video_id',
        'mode',
        'query_hash',
    ])
    .size()
    .rename('actual_candidate_count')
    .reset_index()
)

ready_pairs = success_summary.merge(
    candidate_counts,
    on=['video_id', 'mode', 'query_hash'],
    how='left',
    validate='one_to_one',
)

ready_pairs['actual_candidate_count'] = (
    ready_pairs['actual_candidate_count']
    .fillna(0)
    .astype(int)
)
ready_pairs['eval_ready'] = (
    ready_pairs['actual_candidate_count'].gt(0)
)

completion_report = ready_pairs[[
    'video_id',
    'mode',
    'query_hash',
    'status',
    'category',
    'split',
    'creator_name',
    'title',
    'body_start_sec',
    'body_end_sec',
    'segment_sec',
    'candidate_count',
    'actual_candidate_count',
    'eval_ready',
]].copy()

ready_pairs = (
    ready_pairs[
        ready_pairs['eval_ready']
    ]
    .reset_index(drop=True)
)

if not selected_raw.empty:
    selected = selected_raw.merge(
        ready_pairs[[
            'video_id',
            'mode',
            'query_hash',
        ]],
        on=['video_id', 'mode', 'query_hash'],
        how='inner',
        validate='many_to_one',
    )
else:
    selected = pd.DataFrame()

if ready_pairs.empty:
    raise RuntimeError(
        '평가 가능한 성공 결과가 없습니다.'
    )

mode_counts = (
    ready_pairs
    .groupby('video_id')['mode']
    .nunique()
)
paired_video_ids = set(
    mode_counts[
        mode_counts == len(EXPECTED_MODES)
    ].index
)

if REQUIRE_COMPLETE:
    incomplete = sorted(
        set(ready_pairs['video_id'])
        - paired_video_ids
    )
    if incomplete:
        raise RuntimeError(
            'Fixed/Individual 쌍이 완성되지 않은 영상: '
            f'{incomplete[:20]}'
        )

print('=' * 80)
print('추론 결과 로드 완료')
print('=' * 80)
print('최신 성공 영상-모드:', len(success_summary))
print('평가 가능 영상-모드:', len(ready_pairs))
print(
    '평가 가능 고유 영상:',
    ready_pairs['video_id'].nunique(),
)
print(
    'Fixed/Individual 쌍 완성 영상:',
    len(paired_video_ids),
)
print('후보 행:', len(candidates))

display(
    ready_pairs
    .groupby(['split', 'category', 'mode'])
    .size()
    .reset_index(name='pair_count')
)

## 3. Heatmap 자동 연결

In [ ]:
known_video_ids = set(ready_pairs['video_id'])

heatmap_candidates = sorted(
    path
    for path in HEATMAP_ROOT.rglob('*_heatmap_seconds*.csv')
    if 'merged' not in path.name.lower()
)

heatmap_matches = defaultdict(list)

for path in heatmap_candidates:
    filename = normalize_text(path.name)
    hits = [
        video_id
        for video_id in known_video_ids
        if video_id in filename
    ]
    if len(hits) == 1:
        heatmap_matches[hits[0]].append(path)

duplicate_heatmaps = {
    video_id: paths
    for video_id, paths in heatmap_matches.items()
    if len(paths) > 1
}

if duplicate_heatmaps:
    duplicate_rows = [
        {
            'video_id': video_id,
            'path': str(path),
        }
        for video_id, paths in duplicate_heatmaps.items()
        for path in paths
    ]
    display(pd.DataFrame(duplicate_rows))
    raise ValueError(
        '동일 video_id의 Heatmap 파일이 여러 개입니다.'
    )

heatmap_path_map = {
    video_id: paths[0]
    for video_id, paths in heatmap_matches.items()
    if len(paths) == 1
}

missing_heatmap_ids = sorted(
    known_video_ids - set(heatmap_path_map)
)

heatmap_missing_report = ready_pairs[
    ready_pairs['video_id'].isin(
        missing_heatmap_ids
    )
][[
    'video_id',
    'mode',
    'split',
    'category',
    'creator_name',
    'title',
]].copy()

if missing_heatmap_ids:
    print(
        f'주의: Heatmap이 없는 영상 '
        f'{len(missing_heatmap_ids)}개는 제외합니다.'
    )
    display(heatmap_missing_report.head(30))

ready_pairs = (
    ready_pairs[
        ~ready_pairs['video_id'].isin(
            missing_heatmap_ids
        )
    ]
    .reset_index(drop=True)
)

ready_keys_after_heatmap = ready_pairs[[
    'video_id',
    'mode',
    'query_hash',
]]

candidates = candidates.merge(
    ready_keys_after_heatmap,
    on=['video_id', 'mode', 'query_hash'],
    how='inner',
    validate='many_to_one',
)

if not selected.empty:
    selected = selected.merge(
        ready_keys_after_heatmap,
        on=['video_id', 'mode', 'query_hash'],
        how='inner',
        validate='many_to_one',
    )

known_video_ids = set(ready_pairs['video_id'])
heatmaps = {}
heatmap_manifest_rows = []

for video_id in sorted(known_video_ids):
    path = heatmap_path_map[video_id]
    frame = pd.read_csv(
        path,
        encoding='utf-8-sig',
    )
    frame.columns = [
        normalize_column_name(column)
        for column in frame.columns
    ]

    require_columns(
        frame,
        ['video_id', 'second', 'heatmap_score'],
        f'Heatmap {path.name}',
    )

    internal_ids = {
        normalize_text(value)
        for value in frame[
            'video_id'
        ].dropna().unique()
    }

    if internal_ids != {video_id}:
        raise ValueError(
            f'{path.name}: 내부 video_id 불일치 '
            f'{internal_ids} != {{{video_id}}}'
        )

    clean = pd.DataFrame({
        'second': pd.to_numeric(
            frame['second'],
            errors='coerce',
        ),
        'heatmap_score': pd.to_numeric(
            frame['heatmap_score'],
            errors='coerce',
        ),
    }).dropna()

    clean = (
        clean
        .groupby(
            'second',
            as_index=False,
        )['heatmap_score']
        .mean()
        .sort_values('second')
        .reset_index(drop=True)
    )

    if clean.empty:
        raise ValueError(
            f'{path.name}: 유효한 Heatmap 행이 없습니다.'
        )

    heatmaps[video_id] = clean
    heatmap_manifest_rows.append({
        'video_id': video_id,
        'heatmap_path': str(path),
        'heatmap_rows': len(clean),
        'heatmap_start_sec': float(
            clean['second'].min()
        ),
        'heatmap_end_sec': float(
            clean['second'].max()
        ),
    })

heatmap_manifest = pd.DataFrame(
    heatmap_manifest_rows
)

print(
    '평가에 연결된 Heatmap:',
    len(heatmaps),
)
print(
    '최종 평가 영상-모드:',
    len(ready_pairs),
)
display(heatmap_manifest.head())

## 4. MR 평가·후보 선택 공통 함수

In [ ]:
def temporal_iou(interval_a, interval_b) -> float:
    start_a, end_a = interval_a
    start_b, end_b = interval_b
    intersection = max(
        0.0,
        min(end_a, end_b) - max(start_a, start_b),
    )
    union = (
        (end_a - start_a)
        + (end_b - start_b)
        - intersection
    )
    return float(intersection / union) if union > 0 else 0.0


def one_to_one_matches(predicted, truth, threshold):
    match_candidates = []

    for predicted_index, predicted_interval in enumerate(predicted):
        for truth_index, truth_interval in enumerate(truth):
            score = temporal_iou(
                predicted_interval,
                truth_interval,
            )
            if score >= threshold:
                match_candidates.append(
                    (score, predicted_index, truth_index)
                )

    match_candidates.sort(reverse=True)
    used_prediction = set()
    used_truth = set()
    matches = []

    for score, predicted_index, truth_index in match_candidates:
        if (
            predicted_index in used_prediction
            or truth_index in used_truth
        ):
            continue

        used_prediction.add(predicted_index)
        used_truth.add(truth_index)
        matches.append(
            (predicted_index, truth_index, float(score))
        )

    return matches


def interval_heatmap_mean(heatmap, start_sec, end_sec) -> float:
    seconds = heatmap['second'].to_numpy(dtype=float)
    scores = heatmap['heatmap_score'].to_numpy(dtype=float)

    overlap = (
        np.minimum(seconds + 1.0, end_sec)
        - np.maximum(seconds, start_sec)
    )
    overlap = np.clip(overlap, 0.0, None)
    total = overlap.sum()

    if total <= 0:
        midpoint = (start_sec + end_sec) / 2.0
        return float(
            np.interp(midpoint, seconds, scores)
        )

    return float(
        np.sum(scores * overlap) / total
    )


def context_window(
    anchor_start,
    anchor_end,
    body_duration,
    before,
    after,
):
    center = (
        anchor_start + anchor_end
    ) / 2.0
    segment_length = before + after

    start = center - before
    end = center + after

    if start < 0:
        start = 0.0
        end = min(
            body_duration,
            segment_length,
        )

    if end > body_duration:
        end = body_duration
        start = max(
            0.0,
            body_duration - segment_length,
        )

    return float(start), float(end)


def scaled_offsets(category, actual_segment_sec):
    settings = CATEGORY_SETTINGS[category]
    base_before = float(settings['before_sec'])
    base_after = float(settings['after_sec'])
    base_total = base_before + base_after

    if actual_segment_sec <= 0:
        raise ValueError(
            f'segment_sec가 0 이하입니다: '
            f'{actual_segment_sec}'
        )

    scale = float(actual_segment_sec) / base_total
    return (
        base_before * scale,
        base_after * scale,
    )


def intervals_conflict(
    start,
    end,
    selected_rows,
    min_gap,
):
    return any(
        start < old['end_full_sec'] + min_gap
        and end > old['start_full_sec'] - min_gap
        for old in selected_rows
    )


def select_model_topk(
    candidate_frame,
    top_k,
    min_gap_sec,
    pool_n=None,
):
    # 추론 노트북의 3단계 선택 규칙을 재현합니다.
    frame = candidate_frame.sort_values(
        ['candidate_rank', 'anchor_clip_index']
    ).copy()

    if pool_n is not None:
        frame = frame[
            frame['candidate_rank'] <= int(pool_n)
        ].copy()

    selected_rows = []
    selected_anchors = set()

    def add_pass(
        gap,
        pass_name,
        allow_overlap=False,
    ):
        for row in frame.to_dict('records'):
            anchor = int(row['anchor_clip_index'])

            if anchor in selected_anchors:
                continue

            if (
                not allow_overlap
                and intervals_conflict(
                    float(row['start_full_sec']),
                    float(row['end_full_sec']),
                    selected_rows,
                    float(gap),
                )
            ):
                continue

            current = dict(row)
            current['selection_pass_v2'] = pass_name
            selected_rows.append(current)
            selected_anchors.add(anchor)

            if len(selected_rows) >= int(top_k):
                return

    add_pass(
        min_gap_sec,
        'category_gap',
    )

    if len(selected_rows) < int(top_k):
        add_pass(
            0.0,
            'non_overlap',
        )

    if len(selected_rows) < int(top_k):
        add_pass(
            0.0,
            'overlap_fallback',
            allow_overlap=True,
        )

    selected_frame = pd.DataFrame(
        selected_rows[:int(top_k)]
    )

    if not selected_frame.empty:
        selected_frame['selection_rank_v2'] = np.arange(
            1,
            len(selected_frame) + 1,
        )

    return selected_frame


def select_oracle_topk(
    candidate_frame,
    top_k,
    min_gap_sec,
):
    # 후보군 안에서 MR 점수를 안다고 가정한 진단용 상한입니다.
    frame = candidate_frame.sort_values(
        ['mr_score', 'candidate_rank'],
        ascending=[False, True],
    ).copy()

    frame = frame.reset_index(drop=True)
    frame['_oracle_rank'] = np.arange(
        1,
        len(frame) + 1,
    )
    frame['candidate_rank_original'] = (
        frame['candidate_rank']
    )
    frame['candidate_rank'] = frame['_oracle_rank']

    selected_frame = select_model_topk(
        frame,
        top_k=top_k,
        min_gap_sec=min_gap_sec,
        pool_n=None,
    )

    if not selected_frame.empty:
        selected_frame['candidate_rank'] = (
            selected_frame['candidate_rank_original']
        )

    return selected_frame


_MR_CANDIDATE_CACHE = {}


def build_mr_candidates(
    video_id,
    body_start,
    body_end,
    category,
    actual_segment_sec,
):
    cache_key = (
        video_id,
        round(float(body_start), 6),
        round(float(body_end), 6),
        category,
        round(float(actual_segment_sec), 6),
    )

    if cache_key in _MR_CANDIDATE_CACHE:
        return _MR_CANDIDATE_CACHE[
            cache_key
        ].copy()

    heatmap = heatmaps[video_id]
    settings = CATEGORY_SETTINGS[category]
    body_duration = float(
        body_end - body_start
    )
    clip_sec = float(
        settings['analysis_clip_sec']
    )
    before_sec, after_sec = scaled_offsets(
        category,
        actual_segment_sec,
    )

    rows = []
    anchor_start = 0.0
    anchor_index = 0

    while anchor_start < body_duration:
        anchor_end = min(
            anchor_start + clip_sec,
            body_duration,
        )

        start_body, end_body = context_window(
            anchor_start,
            anchor_end,
            body_duration,
            before_sec,
            after_sec,
        )

        start_full = float(
            body_start + start_body
        )
        end_full = float(
            body_start + end_body
        )

        rows.append({
            'candidate_rank': anchor_index + 1,
            'anchor_clip_index': anchor_index,
            'start_full_sec': start_full,
            'end_full_sec': end_full,
            'duration_sec': (
                end_full - start_full
            ),
            'mr_score': interval_heatmap_mean(
                heatmap,
                start_full,
                end_full,
            ),
        })

        anchor_start += clip_sec
        anchor_index += 1

    result = pd.DataFrame(rows)
    _MR_CANDIDATE_CACHE[
        cache_key
    ] = result.copy()

    return result


def build_mr_ideal_intervals(
    video_id,
    body_start,
    body_end,
    category,
    actual_segment_sec,
    top_k,
):
    mr_candidates = build_mr_candidates(
        video_id,
        body_start,
        body_end,
        category,
        actual_segment_sec,
    )

    selected_frame = select_oracle_topk(
        mr_candidates,
        top_k=int(top_k),
        min_gap_sec=float(
            CATEGORY_SETTINGS[
                category
            ]['min_gap_sec']
        ),
    )

    return (
        selected_frame
        .sort_values('selection_rank_v2')
        .reset_index(drop=True)
    )


def pad_to_k(values, k, fill_value=0.0):
    values = list(values)[:int(k)]

    if len(values) < int(k):
        values.extend(
            [fill_value]
            * (int(k) - len(values))
        )

    return np.asarray(
        values,
        dtype=float,
    )


def ndcg_at_k(
    relevance,
    ideal_relevance,
    k,
) -> float:
    relevance = pad_to_k(
        relevance,
        k,
        0.0,
    )
    ideal_relevance = pad_to_k(
        ideal_relevance,
        k,
        0.0,
    )

    discounts = 1.0 / np.log2(
        np.arange(2, int(k) + 2)
    )

    dcg = np.sum(
        (
            np.power(2.0, relevance)
            - 1.0
        )
        * discounts
    )
    idcg = np.sum(
        (
            np.power(2.0, ideal_relevance)
            - 1.0
        )
        * discounts
    )

    return (
        float(dcg / idcg)
        if idcg > 0
        else np.nan
    )


def evaluate_mr_intervals(
    video_id,
    predicted,
    ideal_frame,
    body_start,
    body_end,
    requested_k,
):
    heatmap = heatmaps[video_id]

    predicted = [
        (float(start), float(end))
        for start, end in predicted
    ]

    ideal_intervals = list(zip(
        ideal_frame[
            'start_full_sec'
        ].astype(float),
        ideal_frame[
            'end_full_sec'
        ].astype(float),
    ))
    ideal_scores = (
        ideal_frame['mr_score']
        .astype(float)
        .tolist()
    )

    mr_scores = [
        interval_heatmap_mean(
            heatmap,
            start,
            end,
        )
        for start, end in predicted
    ]

    matches = one_to_one_matches(
        predicted,
        ideal_intervals,
        MR_TIOU_THRESHOLD,
    )

    pairwise_tious = [
        temporal_iou(
            predicted[left],
            predicted[right],
        )
        for left in range(len(predicted))
        for right in range(
            left + 1,
            len(predicted),
        )
    ]

    global_mean = interval_heatmap_mean(
        heatmap,
        body_start,
        body_end,
    )

    selected_mean = (
        float(np.mean(mr_scores))
        if mr_scores
        else np.nan
    )
    ideal_mean = (
        float(np.mean(ideal_scores))
        if ideal_scores
        else np.nan
    )

    ideal_count = len(ideal_intervals)
    selected_count = len(predicted)
    ndcg_value = ndcg_at_k(
        mr_scores,
        ideal_scores,
        int(requested_k),
    )

    return {
        'requested_k': int(requested_k),
        'selected_count': int(selected_count),
        'ideal_count': int(ideal_count),
        'mr_ndcg_at_k': ndcg_value,
        'mr_score_100': (
            100.0 * ndcg_value
        ),
        'mr_mean_selected': selected_mean,
        'mr_top1': (
            float(mr_scores[0])
            if mr_scores
            else np.nan
        ),
        'mr_global_mean': float(global_mean),
        'mr_lift_over_global': (
            selected_mean
            - float(global_mean)
            if np.isfinite(selected_mean)
            else np.nan
        ),
        'mr_ideal_mean': ideal_mean,
        'mr_regret': (
            ideal_mean - selected_mean
            if (
                np.isfinite(ideal_mean)
                and np.isfinite(selected_mean)
            )
            else np.nan
        ),
        'mr_hits': int(len(matches)),
        'mr_precision_at_k': (
            float(
                len(matches)
                / int(requested_k)
            )
            if requested_k > 0
            else np.nan
        ),
        'mr_recall_at_k': (
            float(
                len(matches)
                / ideal_count
            )
            if ideal_count > 0
            else np.nan
        ),
        'redundancy': (
            float(np.mean(pairwise_tious))
            if pairwise_tious
            else 0.0
        ),
    }


def fixed_length_interval(
    center,
    body_start,
    body_end,
    segment_sec,
):
    body_duration = body_end - body_start
    length = min(
        float(segment_sec),
        float(body_duration),
    )

    start = center - length / 2.0
    end = center + length / 2.0

    if start < body_start:
        start = body_start
        end = body_start + length

    if end > body_end:
        end = body_end
        start = body_end - length

    return float(start), float(end)


def build_uniform_intervals(
    body_start,
    body_end,
    segment_sec,
    top_k,
):
    body_duration = float(
        body_end - body_start
    )

    centers = body_start + (
        (
            np.arange(
                int(top_k),
                dtype=float,
            )
            + 0.5
        )
        / int(top_k)
    ) * body_duration

    return [
        fixed_length_interval(
            center,
            body_start,
            body_end,
            segment_sec,
        )
        for center in centers
    ]


def build_early_intervals(
    body_start,
    body_end,
    segment_sec,
    top_k,
    min_gap_sec,
):
    segment_sec = min(
        float(segment_sec),
        float(body_end - body_start),
    )
    step = (
        segment_sec
        + float(min_gap_sec)
    )
    intervals = []

    for index in range(int(top_k)):
        start = (
            body_start
            + index * step
        )

        if start + segment_sec > body_end:
            break

        intervals.append(
            (
                float(start),
                float(start + segment_sec),
            )
        )

    if len(intervals) < int(top_k):
        return build_uniform_intervals(
            body_start,
            body_end,
            segment_sec,
            int(top_k),
        )

    return intervals


def build_random_intervals(
    body_start,
    body_end,
    segment_sec,
    top_k,
    min_gap_sec,
    rng,
):
    body_duration = float(
        body_end - body_start
    )
    segment_sec = min(
        float(segment_sec),
        body_duration,
    )
    max_start = body_end - segment_sec

    if max_start <= body_start:
        return [
            (
                float(body_start),
                float(body_end),
            )
        ]

    for gap in [
        float(min_gap_sec),
        0.0,
    ]:
        chosen = []
        attempts = 0

        while (
            len(chosen) < int(top_k)
            and attempts < 5000
        ):
            attempts += 1

            start = float(
                rng.uniform(
                    body_start,
                    max_start,
                )
            )
            end = start + segment_sec

            too_close = any(
                start < old_end + gap
                and end > old_start - gap
                for old_start, old_end in chosen
            )

            if not too_close:
                chosen.append(
                    (start, end)
                )

        if len(chosen) == int(top_k):
            rng.shuffle(chosen)
            return chosen

    return build_uniform_intervals(
        body_start,
        body_end,
        segment_sec,
        int(top_k),
    )

## 5. 공식 Top-4 평가와 저장 결과 재현 검증

In [ ]:
official_rows = []
official_segment_rows = []
selection_consistency_rows = []

for pair in ready_pairs.itertuples(index=False):
    current_candidates = candidates[
        (candidates['video_id'] == pair.video_id)
        & (candidates['mode'] == pair.mode)
        & (candidates['query_hash'] == pair.query_hash)
    ].sort_values('candidate_rank').copy()

    min_gap_sec = float(
        CATEGORY_SETTINGS[pair.category]['min_gap_sec']
    )

    reconstructed = select_model_topk(
        current_candidates,
        top_k=OFFICIAL_K,
        min_gap_sec=min_gap_sec,
        pool_n=max(CANDIDATE_NS),
    )

    saved_current = pd.DataFrame()

    if not selected.empty:
        saved_current = selected[
            (selected['video_id'] == pair.video_id)
            & (selected['mode'] == pair.mode)
            & (selected['query_hash'] == pair.query_hash)
        ].copy()

        saved_rank_column = (
            'score_rank'
            if 'score_rank' in saved_current.columns
            else 'final_rank'
        )

        if saved_rank_column in saved_current.columns:
            saved_current = saved_current.sort_values(
                saved_rank_column
            )

    use_saved = len(saved_current) > 0
    final_selection = (
        saved_current.copy()
        if use_saved
        else reconstructed.copy()
    )

    reconstructed_anchors = (
        reconstructed['anchor_clip_index']
        .dropna()
        .astype(int)
        .tolist()
        if not reconstructed.empty
        else []
    )

    saved_anchors = (
        saved_current['anchor_clip_index']
        .dropna()
        .astype(int)
        .tolist()
        if (
            not saved_current.empty
            and 'anchor_clip_index'
            in saved_current.columns
        )
        else []
    )

    selection_match = (
        saved_anchors == reconstructed_anchors
        if saved_anchors
        else np.nan
    )

    selection_consistency_rows.append({
        'video_id': pair.video_id,
        'mode': pair.mode,
        'query_hash': pair.query_hash,
        'saved_selected_count': len(saved_current),
        'reconstructed_count': len(reconstructed),
        'saved_anchor_sequence': '|'.join(
            map(str, saved_anchors)
        ),
        'reconstructed_anchor_sequence': '|'.join(
            map(str, reconstructed_anchors)
        ),
        'saved_equals_reconstructed': selection_match,
        'evaluation_source': (
            'saved_selected_segments'
            if use_saved
            else 'reconstructed_from_top50'
        ),
    })

    ideal = build_mr_ideal_intervals(
        pair.video_id,
        pair.body_start_sec,
        pair.body_end_sec,
        pair.category,
        pair.segment_sec,
        OFFICIAL_K,
    )

    predicted = list(zip(
        final_selection['start_full_sec'],
        final_selection['end_full_sec'],
    ))

    result = evaluate_mr_intervals(
        pair.video_id,
        predicted,
        ideal,
        pair.body_start_sec,
        pair.body_end_sec,
        OFFICIAL_K,
    )

    official_rows.append({
        'video_id': pair.video_id,
        'creator_name': pair.creator_name,
        'title': pair.title,
        'category': pair.category,
        'split': pair.split,
        'mode': pair.mode,
        'query_hash': pair.query_hash,
        'available_candidate_count': len(current_candidates),
        'selection_source': (
            'saved_selected_segments'
            if use_saved
            else 'reconstructed_from_top50'
        ),
        'selected_duration_sec': float(
            final_selection['duration_sec'].sum()
        ),
        'mr_ndcg_at_4': result['mr_ndcg_at_k'],
        'mr_primary_score_100': result['mr_score_100'],
        'mr_mean_at_4': result['mr_mean_selected'],
        'mr_top1': result['mr_top1'],
        'mr_global_mean': result['mr_global_mean'],
        'mr_lift_over_global': result['mr_lift_over_global'],
        'mr_ideal_mean_at_4': result['mr_ideal_mean'],
        'mr_regret_at_4': result['mr_regret'],
        'mr_hits_at_4': result['mr_hits'],
        'mr_precision_at_4': result['mr_precision_at_k'],
        'mr_recall_at_4': result['mr_recall_at_k'],
        'redundancy': result['redundancy'],
    })

    for rank, segment in enumerate(
        final_selection.to_dict('records'),
        start=1,
    ):
        mr_score = interval_heatmap_mean(
            heatmaps[pair.video_id],
            float(segment['start_full_sec']),
            float(segment['end_full_sec']),
        )

        official_segment_rows.append({
            'video_id': pair.video_id,
            'category': pair.category,
            'split': pair.split,
            'mode': pair.mode,
            'query_hash': pair.query_hash,
            'official_rank': rank,
            'candidate_rank': segment.get(
                'candidate_rank',
                np.nan,
            ),
            'anchor_clip_index': segment.get(
                'anchor_clip_index',
                np.nan,
            ),
            'start_full_sec': float(
                segment['start_full_sec']
            ),
            'end_full_sec': float(
                segment['end_full_sec']
            ),
            'duration_sec': float(
                segment['duration_sec']
            ),
            'combined_score': float(
                segment.get(
                    'combined_score',
                    np.nan,
                )
            ),
            'mr_score': float(mr_score),
            'best_query_index': segment.get(
                'best_query_index',
                np.nan,
            ),
            'best_query': normalize_text(
                segment.get('best_query', '')
            ),
        })

mr_video_scores = pd.DataFrame(official_rows)
mr_evaluated_segments = pd.DataFrame(
    official_segment_rows
)
selection_consistency = pd.DataFrame(
    selection_consistency_rows
)

MR_OFFICIAL_METRICS = [
    'mr_ndcg_at_4',
    'mr_primary_score_100',
    'mr_mean_at_4',
    'mr_top1',
    'mr_global_mean',
    'mr_lift_over_global',
    'mr_ideal_mean_at_4',
    'mr_regret_at_4',
    'mr_precision_at_4',
    'mr_recall_at_4',
    'redundancy',
]

mr_overall_summary = (
    mr_video_scores
    .groupby('mode', dropna=False)
    .agg(
        video_count=('video_id', 'nunique'),
        **{
            column: (column, 'mean')
            for column in MR_OFFICIAL_METRICS
        },
    )
    .reset_index()
)

mr_by_category = (
    mr_video_scores
    .groupby(
        ['category', 'mode'],
        dropna=False,
    )
    .agg(
        video_count=('video_id', 'nunique'),
        **{
            column: (column, 'mean')
            for column in MR_OFFICIAL_METRICS
        },
    )
    .reset_index()
)

mismatch_count = int(
    selection_consistency[
        selection_consistency[
            'saved_equals_reconstructed'
        ].eq(False)
    ].shape[0]
)

print('공식 Top-4 평가 완료')
print('평가 영상-모드:', len(mr_video_scores))
print(
    '저장 Top-4와 재구성 Top-4 불일치:',
    mismatch_count,
)
display(mr_overall_summary)
display(mr_by_category)

## 6. Candidate Recall@4/10/20/50 — 검색 단계 진단

In [ ]:
candidate_recall_rows = []
candidate_pool_sensitivity_rows = []

for pair in ready_pairs.itertuples(index=False):
    current_candidates = candidates[
        (candidates['video_id'] == pair.video_id)
        & (candidates['mode'] == pair.mode)
        & (candidates['query_hash'] == pair.query_hash)
    ].sort_values('candidate_rank').copy()

    min_gap_sec = float(
        CATEGORY_SETTINGS[pair.category]['min_gap_sec']
    )

    ideal4 = build_mr_ideal_intervals(
        pair.video_id,
        pair.body_start_sec,
        pair.body_end_sec,
        pair.category,
        pair.segment_sec,
        OFFICIAL_K,
    )

    ideal_intervals = list(zip(
        ideal4['start_full_sec'].astype(float),
        ideal4['end_full_sec'].astype(float),
    ))

    for candidate_n in CANDIDATE_NS:
        pool = current_candidates[
            current_candidates['candidate_rank']
            <= int(candidate_n)
        ].copy()

        pool['mr_score'] = [
            interval_heatmap_mean(
                heatmaps[pair.video_id],
                float(row.start_full_sec),
                float(row.end_full_sec),
            )
            for row in pool.itertuples(index=False)
        ]

        pool_intervals = list(zip(
            pool['start_full_sec'].astype(float),
            pool['end_full_sec'].astype(float),
        ))

        matches = one_to_one_matches(
            pool_intervals,
            ideal_intervals,
            MR_TIOU_THRESHOLD,
        )

        threshold_ranks = []

        for predicted_index, predicted_interval in enumerate(
            pool_intervals
        ):
            for truth_interval in ideal_intervals:
                if temporal_iou(
                    predicted_interval,
                    truth_interval,
                ) >= MR_TIOU_THRESHOLD:
                    threshold_ranks.append(
                        int(
                            pool.iloc[
                                predicted_index
                            ]['candidate_rank']
                        )
                    )

        first_hit_rank = (
            min(threshold_ranks)
            if threshold_ranks
            else np.nan
        )

        matched_ideal_count = len({
            truth_index
            for _, truth_index, _ in matches
        })

        recall_value = (
            matched_ideal_count / len(ideal_intervals)
            if ideal_intervals
            else np.nan
        )

        oracle_selected = select_oracle_topk(
            pool,
            top_k=OFFICIAL_K,
            min_gap_sec=min_gap_sec,
        )

        oracle_predicted = list(zip(
            oracle_selected['start_full_sec'],
            oracle_selected['end_full_sec'],
        ))

        oracle_result = evaluate_mr_intervals(
            pair.video_id,
            oracle_predicted,
            ideal4,
            pair.body_start_sec,
            pair.body_end_sec,
            OFFICIAL_K,
        )

        candidate_recall_rows.append({
            'video_id': pair.video_id,
            'creator_name': pair.creator_name,
            'title': pair.title,
            'category': pair.category,
            'split': pair.split,
            'mode': pair.mode,
            'query_hash': pair.query_hash,
            'candidate_n_requested': int(candidate_n),
            'candidate_n_used': len(pool),
            'oracle_target_k': len(ideal_intervals),
            'matched_oracle_count': matched_ideal_count,
            'candidate_recall_at_n': recall_value,
            'candidate_any_hit_at_n': float(
                matched_ideal_count > 0
            ),
            'candidate_full_recall_at_n': float(
                matched_ideal_count
                == len(ideal_intervals)
                and len(ideal_intervals) > 0
            ),
            'first_hit_candidate_rank': first_hit_rank,
            'candidate_oracle_ndcg_at_4': (
                oracle_result['mr_ndcg_at_k']
            ),
            'candidate_oracle_mean_at_4': (
                oracle_result['mr_mean_selected']
            ),
            'candidate_oracle_hits_at_4': (
                oracle_result['mr_hits']
            ),
        })

        model_selected = select_model_topk(
            current_candidates,
            top_k=OFFICIAL_K,
            min_gap_sec=min_gap_sec,
            pool_n=int(candidate_n),
        )

        model_predicted = list(zip(
            model_selected['start_full_sec'],
            model_selected['end_full_sec'],
        ))

        model_result = evaluate_mr_intervals(
            pair.video_id,
            model_predicted,
            ideal4,
            pair.body_start_sec,
            pair.body_end_sec,
            OFFICIAL_K,
        )

        candidate_pool_sensitivity_rows.append({
            'video_id': pair.video_id,
            'creator_name': pair.creator_name,
            'title': pair.title,
            'category': pair.category,
            'split': pair.split,
            'mode': pair.mode,
            'query_hash': pair.query_hash,
            'candidate_n_requested': int(candidate_n),
            'candidate_n_used': len(pool),
            'final_k': OFFICIAL_K,
            'selected_count': len(model_selected),
            'selected_duration_sec': float(
                model_selected['duration_sec'].sum()
            ),
            'max_selected_candidate_rank': (
                float(
                    model_selected['candidate_rank'].max()
                )
                if not model_selected.empty
                else np.nan
            ),
            **model_result,
        })

candidate_recall_video = pd.DataFrame(
    candidate_recall_rows
)
candidate_pool_sensitivity_video = pd.DataFrame(
    candidate_pool_sensitivity_rows
)

candidate_recall_summary = (
    candidate_recall_video
    .groupby(
        ['mode', 'candidate_n_requested'],
        dropna=False,
    )
    .agg(
        video_count=('video_id', 'nunique'),
        candidate_n_used_mean=(
            'candidate_n_used',
            'mean',
        ),
        candidate_recall_at_n=(
            'candidate_recall_at_n',
            'mean',
        ),
        candidate_any_hit_at_n=(
            'candidate_any_hit_at_n',
            'mean',
        ),
        candidate_full_recall_at_n=(
            'candidate_full_recall_at_n',
            'mean',
        ),
        first_hit_candidate_rank_mean=(
            'first_hit_candidate_rank',
            'mean',
        ),
        candidate_oracle_ndcg_at_4=(
            'candidate_oracle_ndcg_at_4',
            'mean',
        ),
        candidate_oracle_mean_at_4=(
            'candidate_oracle_mean_at_4',
            'mean',
        ),
    )
    .reset_index()
)

candidate_recall_by_category = (
    candidate_recall_video
    .groupby(
        [
            'category',
            'mode',
            'candidate_n_requested',
        ],
        dropna=False,
    )
    .agg(
        video_count=('video_id', 'nunique'),
        candidate_recall_at_n=(
            'candidate_recall_at_n',
            'mean',
        ),
        candidate_any_hit_at_n=(
            'candidate_any_hit_at_n',
            'mean',
        ),
        candidate_full_recall_at_n=(
            'candidate_full_recall_at_n',
            'mean',
        ),
        candidate_oracle_ndcg_at_4=(
            'candidate_oracle_ndcg_at_4',
            'mean',
        ),
    )
    .reset_index()
)

candidate_pool_sensitivity_summary = (
    candidate_pool_sensitivity_video
    .groupby(
        ['mode', 'candidate_n_requested'],
        dropna=False,
    )
    .agg(
        video_count=('video_id', 'nunique'),
        selected_count=('selected_count', 'mean'),
        selected_duration_sec=(
            'selected_duration_sec',
            'mean',
        ),
        mr_ndcg_at_4=('mr_ndcg_at_k', 'mean'),
        mr_mean_at_4=('mr_mean_selected', 'mean'),
        mr_precision_at_4=(
            'mr_precision_at_k',
            'mean',
        ),
        mr_recall_at_4=(
            'mr_recall_at_k',
            'mean',
        ),
        max_selected_candidate_rank=(
            'max_selected_candidate_rank',
            'mean',
        ),
    )
    .reset_index()
)

print('Candidate Recall 요약')
display(candidate_recall_summary)

print('\n카테고리별 Candidate Recall')
display(candidate_recall_by_category)

print('\n후보군 N에 따른 최종 K=4 성능')
display(candidate_pool_sensitivity_summary)

## 7. K=1/2/4/6 민감도 실험

In [ ]:
k_sensitivity_rows = []
k_selected_segment_rows = []

for pair in ready_pairs.itertuples(index=False):
    current_candidates = candidates[
        (candidates['video_id'] == pair.video_id)
        & (candidates['mode'] == pair.mode)
        & (candidates['query_hash'] == pair.query_hash)
    ].sort_values('candidate_rank').copy()

    min_gap_sec = float(
        CATEGORY_SETTINGS[pair.category]['min_gap_sec']
    )

    for requested_k in K_VALUES:
        model_selected = select_model_topk(
            current_candidates,
            top_k=int(requested_k),
            min_gap_sec=min_gap_sec,
            pool_n=max(CANDIDATE_NS),
        )

        ideal = build_mr_ideal_intervals(
            pair.video_id,
            pair.body_start_sec,
            pair.body_end_sec,
            pair.category,
            pair.segment_sec,
            int(requested_k),
        )

        predicted = list(zip(
            model_selected['start_full_sec'],
            model_selected['end_full_sec'],
        ))

        result = evaluate_mr_intervals(
            pair.video_id,
            predicted,
            ideal,
            pair.body_start_sec,
            pair.body_end_sec,
            int(requested_k),
        )

        selected_duration = float(
            model_selected['duration_sec'].sum()
        )
        body_duration = float(
            pair.body_end_sec - pair.body_start_sec
        )

        k_sensitivity_rows.append({
            'video_id': pair.video_id,
            'creator_name': pair.creator_name,
            'title': pair.title,
            'category': pair.category,
            'split': pair.split,
            'mode': pair.mode,
            'query_hash': pair.query_hash,
            'requested_k': int(requested_k),
            'available_candidate_count': len(current_candidates),
            'selected_duration_sec': selected_duration,
            'body_coverage_ratio': (
                selected_duration / body_duration
                if body_duration > 0
                else np.nan
            ),
            'max_selected_candidate_rank': (
                float(
                    model_selected['candidate_rank'].max()
                )
                if not model_selected.empty
                else np.nan
            ),
            **result,
        })

        for row in model_selected.itertuples(index=False):
            mr_score = interval_heatmap_mean(
                heatmaps[pair.video_id],
                float(row.start_full_sec),
                float(row.end_full_sec),
            )

            k_selected_segment_rows.append({
                'video_id': pair.video_id,
                'category': pair.category,
                'split': pair.split,
                'mode': pair.mode,
                'query_hash': pair.query_hash,
                'requested_k': int(requested_k),
                'selection_rank': int(
                    row.selection_rank_v2
                ),
                'candidate_rank': int(
                    row.candidate_rank
                ),
                'anchor_clip_index': int(
                    row.anchor_clip_index
                ),
                'selection_pass': normalize_text(
                    row.selection_pass_v2
                ),
                'start_full_sec': float(
                    row.start_full_sec
                ),
                'end_full_sec': float(
                    row.end_full_sec
                ),
                'duration_sec': float(
                    row.duration_sec
                ),
                'combined_score': float(
                    row.combined_score
                ),
                'mr_score': float(mr_score),
                'best_query_index': getattr(
                    row,
                    'best_query_index',
                    np.nan,
                ),
                'best_query': normalize_text(
                    getattr(row, 'best_query', '')
                ),
            })

k_sensitivity_video = pd.DataFrame(
    k_sensitivity_rows
)
k_selected_segments = pd.DataFrame(
    k_selected_segment_rows
)

k_sensitivity_summary = (
    k_sensitivity_video
    .groupby(
        ['mode', 'requested_k'],
        dropna=False,
    )
    .agg(
        video_count=('video_id', 'nunique'),
        selected_count=('selected_count', 'mean'),
        selected_duration_sec=(
            'selected_duration_sec',
            'mean',
        ),
        body_coverage_ratio=(
            'body_coverage_ratio',
            'mean',
        ),
        mr_ndcg_at_k=('mr_ndcg_at_k', 'mean'),
        mr_score_100=('mr_score_100', 'mean'),
        mr_mean_selected=(
            'mr_mean_selected',
            'mean',
        ),
        mr_top1=('mr_top1', 'mean'),
        mr_lift_over_global=(
            'mr_lift_over_global',
            'mean',
        ),
        mr_precision_at_k=(
            'mr_precision_at_k',
            'mean',
        ),
        mr_recall_at_k=(
            'mr_recall_at_k',
            'mean',
        ),
        mr_regret=('mr_regret', 'mean'),
        redundancy=('redundancy', 'mean'),
        max_selected_candidate_rank=(
            'max_selected_candidate_rank',
            'mean',
        ),
    )
    .reset_index()
)

k_sensitivity_by_category = (
    k_sensitivity_video
    .groupby(
        [
            'category',
            'mode',
            'requested_k',
        ],
        dropna=False,
    )
    .agg(
        video_count=('video_id', 'nunique'),
        selected_duration_sec=(
            'selected_duration_sec',
            'mean',
        ),
        mr_ndcg_at_k=('mr_ndcg_at_k', 'mean'),
        mr_mean_selected=(
            'mr_mean_selected',
            'mean',
        ),
        mr_precision_at_k=(
            'mr_precision_at_k',
            'mean',
        ),
        mr_recall_at_k=(
            'mr_recall_at_k',
            'mean',
        ),
        redundancy=('redundancy', 'mean'),
    )
    .reset_index()
)

print('K 민감도 전체 요약')
display(k_sensitivity_summary)

print('\nK 민감도 카테고리별 요약')
display(k_sensitivity_by_category)

## 8. K별 Fixed vs Individual 대응 비교

In [ ]:
pair_index = [
    'video_id',
    'creator_name',
    'title',
    'category',
    'split',
    'requested_k',
]

metric_columns = [
    'mr_ndcg_at_k',
    'mr_mean_selected',
    'mr_precision_at_k',
    'mr_recall_at_k',
    'selected_duration_sec',
]

fixed_k = k_sensitivity_video[
    k_sensitivity_video['mode'] == 'fixed'
][pair_index + metric_columns].copy()

individual_k = k_sensitivity_video[
    k_sensitivity_video['mode'] == 'individual'
][pair_index + metric_columns].copy()

fixed_k = fixed_k.rename(
    columns={
        column: f'fixed_{column}'
        for column in metric_columns
    }
)
individual_k = individual_k.rename(
    columns={
        column: f'individual_{column}'
        for column in metric_columns
    }
)

k_fixed_vs_individual = fixed_k.merge(
    individual_k,
    on=pair_index,
    how='inner',
    validate='one_to_one',
)

for metric in metric_columns:
    k_fixed_vs_individual[f'diff_{metric}'] = (
        k_fixed_vs_individual[
            f'individual_{metric}'
        ]
        - k_fixed_vs_individual[
            f'fixed_{metric}'
        ]
    )

summary_rows = []

for requested_k, frame in k_fixed_vs_individual.groupby(
    'requested_k'
):
    scopes = [
        ('overall', 'all', frame),
    ]
    scopes.extend(
        (
            'category',
            category,
            category_frame,
        )
        for category, category_frame
        in frame.groupby('category')
    )

    for scope, scope_value, scope_frame in scopes:
        differences = scope_frame[
            'diff_mr_ndcg_at_k'
        ].dropna().to_numpy(float)

        if len(differences) == 0:
            continue

        mean_diff, ci_low, ci_high = (
            bootstrap_mean_ci(
                differences,
                iterations=BOOTSTRAP_ITERATIONS,
                seed=(
                    RANDOM_SEED
                    + int(requested_k)
                ),
            )
        )

        summary_rows.append({
            'scope': scope,
            'scope_value': scope_value,
            'requested_k': int(requested_k),
            'paired_video_count': len(differences),
            'fixed_mean': float(
                scope_frame[
                    'fixed_mr_ndcg_at_k'
                ].mean()
            ),
            'individual_mean': float(
                scope_frame[
                    'individual_mr_ndcg_at_k'
                ].mean()
            ),
            'individual_minus_fixed_mean': mean_diff,
            'bootstrap_ci95_low': ci_low,
            'bootstrap_ci95_high': ci_high,
            'individual_win_rate': float(
                np.mean(differences > 1e-9)
            ),
            'fixed_win_rate': float(
                np.mean(differences < -1e-9)
            ),
            'tie_rate': float(
                np.mean(
                    np.abs(differences) <= 1e-9
                )
            ),
        })

k_fixed_vs_individual_summary = pd.DataFrame(
    summary_rows
)

print('K별 Fixed vs Individual nDCG 대응 비교')
display(
    k_fixed_vs_individual_summary.sort_values(
        [
            'scope',
            'scope_value',
            'requested_k',
        ]
    )
)

## 9. 공식 K=4 Baseline 비교

In [ ]:
baseline_rows = []
rng_random = np.random.default_rng(RANDOM_SEED)

primary_pairs = ready_pairs[
    ready_pairs['mode'] == PRIMARY_MODE
].copy()

if primary_pairs.empty:
    raise RuntimeError(
        f'PRIMARY_MODE={PRIMARY_MODE!r} 결과가 없습니다.'
    )


def baseline_payload(result):
    return {
        'mr_ndcg_at_4': result['mr_ndcg_at_k'],
        'mr_primary_score_100': result['mr_score_100'],
        'mr_mean_at_4': result['mr_mean_selected'],
        'mr_top1': result['mr_top1'],
        'mr_global_mean': result['mr_global_mean'],
        'mr_lift_over_global': result['mr_lift_over_global'],
        'mr_ideal_mean_at_4': result['mr_ideal_mean'],
        'mr_regret_at_4': result['mr_regret'],
        'mr_precision_at_4': result['mr_precision_at_k'],
        'mr_recall_at_4': result['mr_recall_at_k'],
        'redundancy': result['redundancy'],
    }


for row in mr_video_scores.itertuples(index=False):
    method = (
        'individual_query'
        if row.mode == 'individual'
        else 'fixed_query'
    )

    baseline_rows.append({
        'video_id': row.video_id,
        'creator_name': row.creator_name,
        'title': row.title,
        'category': row.category,
        'split': row.split,
        'method': method,
        'mr_ndcg_at_4': row.mr_ndcg_at_4,
        'mr_primary_score_100': row.mr_primary_score_100,
        'mr_mean_at_4': row.mr_mean_at_4,
        'mr_top1': row.mr_top1,
        'mr_global_mean': row.mr_global_mean,
        'mr_lift_over_global': row.mr_lift_over_global,
        'mr_ideal_mean_at_4': row.mr_ideal_mean_at_4,
        'mr_regret_at_4': row.mr_regret_at_4,
        'mr_precision_at_4': row.mr_precision_at_4,
        'mr_recall_at_4': row.mr_recall_at_4,
        'redundancy': row.redundancy,
    })

for pair in primary_pairs.itertuples(index=False):
    body_start = float(pair.body_start_sec)
    body_end = float(pair.body_end_sec)
    segment_sec = float(pair.segment_sec)
    min_gap_sec = float(
        CATEGORY_SETTINGS[pair.category]['min_gap_sec']
    )

    ideal = build_mr_ideal_intervals(
        pair.video_id,
        body_start,
        body_end,
        pair.category,
        segment_sec,
        OFFICIAL_K,
    )

    methods = {
        'uniform': build_uniform_intervals(
            body_start,
            body_end,
            segment_sec,
            OFFICIAL_K,
        ),
        'early': build_early_intervals(
            body_start,
            body_end,
            segment_sec,
            OFFICIAL_K,
            min_gap_sec,
        ),
        'oracle_mr': list(zip(
            ideal['start_full_sec'],
            ideal['end_full_sec'],
        )),
    }

    for method, intervals in methods.items():
        result = evaluate_mr_intervals(
            pair.video_id,
            intervals,
            ideal,
            body_start,
            body_end,
            OFFICIAL_K,
        )

        baseline_rows.append({
            'video_id': pair.video_id,
            'creator_name': pair.creator_name,
            'title': pair.title,
            'category': pair.category,
            'split': pair.split,
            'method': method,
            **baseline_payload(result),
        })

    random_results = []

    for _ in range(RANDOM_BASELINE_TRIALS):
        intervals = build_random_intervals(
            body_start,
            body_end,
            segment_sec,
            OFFICIAL_K,
            min_gap_sec,
            rng_random,
        )

        random_results.append(
            evaluate_mr_intervals(
                pair.video_id,
                intervals,
                ideal,
                body_start,
                body_end,
                OFFICIAL_K,
            )
        )

    average_result = {}

    for key in [
        'mr_ndcg_at_k',
        'mr_score_100',
        'mr_mean_selected',
        'mr_top1',
        'mr_global_mean',
        'mr_lift_over_global',
        'mr_ideal_mean',
        'mr_regret',
        'mr_precision_at_k',
        'mr_recall_at_k',
        'redundancy',
    ]:
        average_result[key] = float(
            np.nanmean([
                result[key]
                for result in random_results
            ])
        )

    baseline_rows.append({
        'video_id': pair.video_id,
        'creator_name': pair.creator_name,
        'title': pair.title,
        'category': pair.category,
        'split': pair.split,
        'method': 'random',
        **baseline_payload(average_result),
    })

mr_baseline_video = pd.DataFrame(
    baseline_rows
)

baseline_metrics = [
    'mr_ndcg_at_4',
    'mr_primary_score_100',
    'mr_mean_at_4',
    'mr_top1',
    'mr_global_mean',
    'mr_lift_over_global',
    'mr_ideal_mean_at_4',
    'mr_regret_at_4',
    'mr_precision_at_4',
    'mr_recall_at_4',
    'redundancy',
]

mr_baseline_summary = (
    mr_baseline_video
    .groupby('method', dropna=False)
    .agg(
        video_count=('video_id', 'nunique'),
        **{
            metric: (metric, 'mean')
            for metric in baseline_metrics
        },
    )
    .reset_index()
    .sort_values(
        'mr_ndcg_at_4',
        ascending=False,
    )
    .reset_index(drop=True)
)

mr_baseline_by_category = (
    mr_baseline_video
    .groupby(
        ['category', 'method'],
        dropna=False,
    )
    .agg(
        video_count=('video_id', 'nunique'),
        **{
            metric: (metric, 'mean')
            for metric in baseline_metrics
        },
    )
    .reset_index()
)

primary_pivot = mr_baseline_video.pivot_table(
    index=[
        'video_id',
        'creator_name',
        'title',
        'category',
        'split',
    ],
    columns='method',
    values='mr_ndcg_at_4',
    aggfunc='first',
).reset_index()

comparison_rows = []

for baseline_method in [
    'random',
    'uniform',
    'early',
    'fixed_query',
]:
    if baseline_method not in primary_pivot.columns:
        continue

    valid = primary_pivot[[
        'video_id',
        'category',
        'split',
        'individual_query',
        baseline_method,
    ]].dropna().copy()

    if valid.empty:
        continue

    differences = (
        valid['individual_query']
        - valid[baseline_method]
    ).to_numpy(float)

    mean_diff, ci_low, ci_high = (
        bootstrap_mean_ci(differences)
    )

    comparison_rows.append({
        'comparison': (
            f'individual_query - {baseline_method}'
        ),
        'paired_video_count': len(valid),
        'individual_mean': float(
            valid['individual_query'].mean()
        ),
        'baseline_mean': float(
            valid[baseline_method].mean()
        ),
        'mean_difference': mean_diff,
        'bootstrap_ci95_low': ci_low,
        'bootstrap_ci95_high': ci_high,
        'individual_win_rate': float(
            np.mean(differences > 1e-9)
        ),
        'baseline_win_rate': float(
            np.mean(differences < -1e-9)
        ),
        'tie_rate': float(
            np.mean(
                np.abs(differences) <= 1e-9
            )
        ),
    })

mr_baseline_paired_comparison = pd.DataFrame(
    comparison_rows
)

print('공식 K=4 Baseline 비교')
display(mr_baseline_summary)

print('\nIndividual Query 대응 비교')
display(mr_baseline_paired_comparison)

## 10. Query index별 진단

In [ ]:
if not mr_evaluated_segments.empty:
    query_index_diagnostic = (
        mr_evaluated_segments
        .dropna(subset=['best_query_index'])
        .assign(
            best_query_index=lambda frame: (
                frame['best_query_index']
                .astype(int)
            )
        )
        .groupby(
            [
                'category',
                'mode',
                'best_query_index',
            ],
            dropna=False,
        )
        .agg(
            selection_count=('video_id', 'size'),
            selected_video_count=(
                'video_id',
                'nunique',
            ),
            mean_official_rank=(
                'official_rank',
                'mean',
            ),
            mean_combined_score=(
                'combined_score',
                'mean',
            ),
            mean_mr_score=(
                'mr_score',
                'mean',
            ),
        )
        .reset_index()
        .sort_values(
            [
                'category',
                'mode',
                'mean_mr_score',
            ],
            ascending=[True, True, False],
        )
    )
else:
    query_index_diagnostic = pd.DataFrame()

print('Query index별 선택 횟수와 MR 평균')
display(query_index_diagnostic)

## 11. 공식 Individual Bootstrap 95% CI

In [ ]:
mr_primary_individual = mr_video_scores[
    mr_video_scores['mode'] == PRIMARY_MODE
].copy()


def primary_bootstrap_row(
    frame,
    scope_type,
    scope_value,
):
    values = (
        frame['mr_ndcg_at_4']
        .dropna()
        .to_numpy(float)
    )

    if len(values) == 0:
        return None

    mean_value, ci_low, ci_high = (
        bootstrap_mean_ci(values)
    )

    return {
        'scope_type': scope_type,
        'scope_value': scope_value,
        'video_count': len(values),
        'mr_ndcg_at_4_mean': mean_value,
        'mr_ndcg_at_4_median': float(
            np.median(values)
        ),
        'mr_ndcg_at_4_std': (
            float(values.std(ddof=1))
            if len(values) > 1
            else 0.0
        ),
        'mr_ndcg_ci95_low': ci_low,
        'mr_ndcg_ci95_high': ci_high,
        'mr_score_100_mean': 100.0 * mean_value,
        'mr_score_100_ci95_low': 100.0 * ci_low,
        'mr_score_100_ci95_high': 100.0 * ci_high,
        'mr_mean_at_4': float(
            frame['mr_mean_at_4'].mean()
        ),
        'mr_top1': float(
            frame['mr_top1'].mean()
        ),
        'mr_lift_over_global': float(
            frame['mr_lift_over_global'].mean()
        ),
        'mr_precision_at_4': float(
            frame['mr_precision_at_4'].mean()
        ),
        'mr_recall_at_4': float(
            frame['mr_recall_at_4'].mean()
        ),
        'mr_regret_at_4': float(
            frame['mr_regret_at_4'].mean()
        ),
        'redundancy': float(
            frame['redundancy'].mean()
        ),
    }


primary_summary_rows = []

overall_row = primary_bootstrap_row(
    mr_primary_individual,
    'overall',
    'all',
)

if overall_row:
    primary_summary_rows.append(
        overall_row
    )

for category, frame in mr_primary_individual.groupby(
    'category'
):
    row = primary_bootstrap_row(
        frame,
        'category',
        category,
    )
    if row:
        primary_summary_rows.append(row)

for split, frame in mr_primary_individual.groupby(
    'split'
):
    row = primary_bootstrap_row(
        frame,
        'split',
        split,
    )
    if row:
        primary_summary_rows.append(row)

mr_primary_summary = pd.DataFrame(
    primary_summary_rows
)

print('공식 Individual Query / MR nDCG@4')
display(mr_primary_summary)

## 12. 결과 저장

In [ ]:
OUTPUTS = {
    'completion_report_v2.csv': completion_report,
    'heatmap_missing_report.csv': heatmap_missing_report,
    'heatmap_manifest.csv': heatmap_manifest,
    'selection_consistency.csv': selection_consistency,
    'mr_video_scores_official_top4.csv': mr_video_scores,
    'mr_evaluated_segments_official_top4.csv': mr_evaluated_segments,
    'mr_overall_summary.csv': mr_overall_summary,
    'mr_by_category.csv': mr_by_category,
    'mr_primary_individual.csv': mr_primary_individual,
    'mr_primary_summary.csv': mr_primary_summary,
    'mr_baseline_video.csv': mr_baseline_video,
    'mr_baseline_summary.csv': mr_baseline_summary,
    'mr_baseline_by_category.csv': mr_baseline_by_category,
    'mr_baseline_paired_comparison.csv': mr_baseline_paired_comparison,
    'candidate_recall_video.csv': candidate_recall_video,
    'candidate_recall_summary.csv': candidate_recall_summary,
    'candidate_recall_by_category.csv': candidate_recall_by_category,
    'candidate_pool_sensitivity_video.csv': candidate_pool_sensitivity_video,
    'candidate_pool_sensitivity_summary.csv': candidate_pool_sensitivity_summary,
    'k_sensitivity_video.csv': k_sensitivity_video,
    'k_sensitivity_summary.csv': k_sensitivity_summary,
    'k_sensitivity_by_category.csv': k_sensitivity_by_category,
    'k_selected_segments.csv': k_selected_segments,
    'k_fixed_vs_individual.csv': k_fixed_vs_individual,
    'k_fixed_vs_individual_summary.csv': k_fixed_vs_individual_summary,
    'query_index_diagnostic.csv': query_index_diagnostic,
}

for filename, frame in OUTPUTS.items():
    save_csv_safely(
        frame,
        EVAL_ROOT / filename,
    )

print('\n' + '=' * 80)
print('UniVTG Most Replayed 평가 v2 완료')
print('=' * 80)
print('저장 폴더:', EVAL_ROOT)
print('평가 영상-모드:', len(mr_video_scores))
print(
    '공식 Individual 영상:',
    len(mr_primary_individual),
)

print('\n[공식 Top-4]')
display(mr_primary_summary)

print('\n[Candidate Recall]')
display(candidate_recall_summary)

print('\n[후보군 N 민감도]')
display(candidate_pool_sensitivity_summary)

print('\n[K 민감도]')
display(k_sensitivity_summary)

print('\n[K별 Fixed vs Individual]')
display(k_fixed_vs_individual_summary)

print('\n[Baseline]')
display(mr_baseline_summary)

## 13. 결과 해석 기준

### Candidate Recall@N

- `Recall@20/50`도 낮음  
  → Query 또는 UniVTG 검색 단계가 MR 장면을 후보군에 넣지 못한 것
- `Recall@20/50`은 높은데 공식 nDCG@4가 낮음  
  → 후보는 찾았지만 최종 랭킹·NMS·Query 결합에서 탈락한 것
- `candidate_oracle_ndcg_at_4`가 높음  
  → 후보군 내부 재랭킹 개선 여지가 큼

### 후보군 N 민감도

- N=4보다 N=20/50에서 nDCG@4 상승  
  → 중복 제거와 간격 조건을 만족하려면 넓은 후보군이 필요
- N을 늘려도 거의 변화 없음  
  → 상위 후보가 이미 고정되어 있거나 검색 자체가 병목

### K 민감도

- K=1/2가 K=4와 비슷하면서 길이가 짧음  
  → 더 짧은 인트로 설정 검토 가능
- K=6에서 성능은 조금 오르지만 길이가 과도하게 증가  
  → 제품은 K=4 유지, 연구적으로 균형 근거 확보
- 카테고리마다 최적 K가 다름  
  → 게임·영화·팟캐스트별 K 분리 검토

### Fixed vs Individual

`k_fixed_vs_individual_summary.csv`의 95% CI를 확인합니다.

- CI 전체가 0보다 큼: Individual 우세
- CI가 0 포함: 뚜렷한 차이 없음
- CI 전체가 0보다 작음: Fixed 우세

## 12. Colab 결과 대시보드 및 CSV 저장 확인

아래 셀은 핵심 결과를 Colab에서 표로 보여주고, 저장된 CSV 목록과 경로를 함께 표시합니다.

- 요약 결과: Colab 표로 표시
- 영상별 상세 결과: CSV로 저장
- 모든 CSV: `EVAL_ROOT` 아래에 UTF-8-SIG 형식으로 저장


In [ ]:
from IPython.display import Markdown, display


def show_result_table(title, frame, *, sort_by=None, ascending=True, max_rows=200):
    """핵심 평가 결과를 읽기 좋은 Colab 표로 표시합니다."""
    display(Markdown(f'### {title}'))

    if frame is None or frame.empty:
        print('표시할 결과가 없습니다.')
        return

    view = frame.copy()

    if sort_by is not None:
        valid_sort_columns = [
            column for column in sort_by
            if column in view.columns
        ]
        if valid_sort_columns:
            view = view.sort_values(
                valid_sort_columns,
                ascending=ascending,
            )

    numeric_columns = view.select_dtypes(
        include=[np.number]
    ).columns
    view[numeric_columns] = view[numeric_columns].round(4)

    if len(view) > max_rows:
        print(
            f'전체 {len(view):,}행 중 상위 '
            f'{max_rows:,}행만 표시합니다. '
            '전체 내용은 CSV를 확인하세요.'
        )
        view = view.head(max_rows)

    with pd.option_context(
        'display.max_columns', None,
        'display.width', 220,
        'display.max_colwidth', 120,
    ):
        display(view.reset_index(drop=True))


# ------------------------------------------------------------
# 핵심 요약표
# ------------------------------------------------------------
show_result_table(
    '1. 공식 Individual Query — MR nDCG@4',
    mr_primary_summary,
)

show_result_table(
    '2. Fixed·Individual·Baseline 비교',
    mr_baseline_summary,
    sort_by=['mr_ndcg_at_4'],
    ascending=False,
)

show_result_table(
    '3. Candidate Recall@4/10/20/50',
    candidate_recall_summary,
    sort_by=['mode', 'candidate_n'],
)

show_result_table(
    '4. 카테고리별 Candidate Recall',
    candidate_recall_by_category,
    sort_by=['category', 'mode', 'candidate_n'],
)

show_result_table(
    '5. 후보군 크기 N 민감도',
    candidate_pool_sensitivity_summary,
    sort_by=['mode', 'candidate_n'],
)

show_result_table(
    '6. K=1/2/4/6 민감도',
    k_sensitivity_summary,
    sort_by=['mode', 'requested_k'],
)

show_result_table(
    '7. 카테고리별 K 민감도',
    k_sensitivity_by_category,
    sort_by=['category', 'mode', 'requested_k'],
)

show_result_table(
    '8. K별 Fixed vs Individual 대응 비교',
    k_fixed_vs_individual_summary,
    sort_by=['scope', 'scope_value', 'requested_k'],
)

show_result_table(
    '9. Query Index별 선택 횟수와 MR 평균',
    query_index_diagnostic,
    sort_by=['category', 'mode', 'mean_mr_score'],
    ascending=[True, True, False],
)


# ------------------------------------------------------------
# 저장된 CSV 파일 목록
# ------------------------------------------------------------
csv_manifest_rows = []

for filename, frame in OUTPUTS.items():
    csv_path = EVAL_ROOT / filename
    csv_manifest_rows.append({
        'filename': filename,
        'rows': 0 if frame is None else len(frame),
        'columns': 0 if frame is None else len(frame.columns),
        'saved': csv_path.exists(),
        'path': str(csv_path),
        'size_kb': (
            round(csv_path.stat().st_size / 1024, 1)
            if csv_path.exists()
            else np.nan
        ),
    })

csv_manifest = pd.DataFrame(csv_manifest_rows)

show_result_table(
    '10. 저장된 CSV 파일 목록',
    csv_manifest,
    sort_by=['filename'],
    max_rows=500,
)

print('\n' + '=' * 80)
print('Colab 표 출력 및 CSV 저장 확인 완료')
print('=' * 80)
print('CSV 저장 폴더:', EVAL_ROOT)
print('저장 완료 파일:', int(csv_manifest['saved'].sum()), '/', len(csv_manifest))
